# Transformer dan Attention Basics

Notebook ini merangkum fondasi `01-dasar-transformers-dan-llm.md`: token sederhana, embedding, Q/K/V, scaled dot-product attention, positional encoding, dan decoding.

In [ ]:
import math
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(7)

## API LLM Opsional

Notebook ini tetap bisa jalan tanpa API key. Jika ingin memanggil contoh LLM, isi `GEMINI_API_KEY` melalui Kaggle Secrets atau environment variable. API key bisa dibuat dari Google AI Studio; free tier memiliki kuota dan dapat berubah.

In [ ]:
import os
import json
import urllib.request
import urllib.error

def get_secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        value = UserSecretsClient().get_secret(name)
        if value:
            return value
    except Exception:
        pass
    return os.getenv(name, "")

GEMINI_API_KEY = get_secret("GEMINI_API_KEY") or "PASTE_GEMINI_API_KEY_HERE"
GEMINI_MODEL = os.getenv("GEMINI_MODEL", "gemini-2.5-flash")

def call_gemini_text(prompt, model=GEMINI_MODEL):
    if not GEMINI_API_KEY or GEMINI_API_KEY.startswith("PASTE_"):
        return "[SKIP] Isi GEMINI_API_KEY di Kaggle Secrets atau environment variable untuk memanggil LLM."

    url = f"https://generativelanguage.googleapis.com/v1beta/models/{model}:generateContent"
    payload = {"contents": [{"parts": [{"text": prompt}]}]}
    request = urllib.request.Request(
        url,
        data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json", "x-goog-api-key": GEMINI_API_KEY},
        method="POST",
    )

    try:
        with urllib.request.urlopen(request, timeout=60) as response:
            data = json.loads(response.read().decode("utf-8"))
    except urllib.error.HTTPError as error:
        detail = error.read().decode("utf-8")[:500]
        return f"[ERROR] Gemini API {error.code}: {detail}"

    return data["candidates"][0]["content"]["parts"][0]["text"]

## 1. Token dan Embedding

Contoh ini memakai token buatan agar fokusnya tetap pada bentuk data yang masuk ke Transformer.

In [ ]:
tokens = ["saya", "belajar", "attention", "hari", "ini"]
token_to_id = {token: index for index, token in enumerate(tokens)}
ids = torch.tensor([token_to_id[token] for token in tokens])

embedding_dim = 6
embedding_table = torch.randn(len(tokens), embedding_dim)
x = embedding_table[ids]

print("Token IDs:", ids.tolist())
print("Shape embedding:", tuple(x.shape))

## 2. Query, Key, Value

Setiap token diproyeksikan menjadi Query, Key, dan Value. Query mencari informasi, Key menjadi label pencocokan, dan Value membawa isi informasi.

In [ ]:
d_k = 4
w_q = torch.randn(embedding_dim, d_k)
w_k = torch.randn(embedding_dim, d_k)
w_v = torch.randn(embedding_dim, d_k)

q = x @ w_q
k = x @ w_k
v = x @ w_v

print("Q:", tuple(q.shape))
print("K:", tuple(k.shape))
print("V:", tuple(v.shape))

## 3. Scaled Dot-Product Attention

Skor attention dihitung dari `QK^T`, lalu dinormalisasi dengan `softmax`. Pembagian dengan `sqrt(d_k)` menjaga skala skor tetap stabil.

In [ ]:
scores = (q @ k.T) / math.sqrt(d_k)
attention_weights = F.softmax(scores, dim=-1)
context = attention_weights @ v

print("Attention weights:")
print(attention_weights.round(decimals=3))
print("Context shape:", tuple(context.shape))

In [ ]:
plt.figure(figsize=(6, 4))
plt.imshow(attention_weights.detach().numpy(), cmap="Blues")
plt.xticks(range(len(tokens)), tokens, rotation=35)
plt.yticks(range(len(tokens)), tokens)
plt.colorbar(label="attention")
plt.title("Token memperhatikan token lain")
plt.tight_layout()
plt.show()

## 4. Positional Encoding

Transformer memproses token secara paralel. Karena itu, posisi token perlu ditambahkan secara eksplisit.

In [ ]:
def sinusoidal_position_encoding(seq_len, dim):
    positions = torch.arange(seq_len).unsqueeze(1)
    div_terms = torch.exp(torch.arange(0, dim, 2) * (-math.log(10000.0) / dim))
    pe = torch.zeros(seq_len, dim)
    pe[:, 0::2] = torch.sin(positions * div_terms)
    pe[:, 1::2] = torch.cos(positions * div_terms)
    return pe

pe = sinusoidal_position_encoding(seq_len=len(tokens), dim=embedding_dim)
x_with_position = x + pe

print("Positional encoding shape:", tuple(pe.shape))
print(pe.round(decimals=3))

## 5. Decoding Sederhana

LLM menghasilkan distribusi probabilitas token. Strategi decoding menentukan token mana yang dipilih dari distribusi itu.

In [ ]:
vocab = ["kopi", "teh", "air", "buku", "kode"]
logits = torch.tensor([2.4, 2.1, 1.2, 0.4, 0.3])

def probabilities_with_temperature(logits, temperature):
    return F.softmax(logits / temperature, dim=-1)

for temperature in [0.4, 1.0, 1.8]:
    probs = probabilities_with_temperature(logits, temperature)
    pairs = list(zip(vocab, probs.round(decimals=3).tolist()))
    print(f"temperature={temperature}:", pairs)

## 6. Contoh Pemanggilan LLM

Cell berikut meminta LLM menjelaskan hasil attention dan decoding. Jika `GEMINI_API_KEY` belum diisi, cell hanya mengembalikan pesan skip.

In [ ]:
attention_summary = attention_weights.round(decimals=3).tolist()
llm_prompt = f"""
Jelaskan konsep attention untuk mahasiswa pemula berdasarkan matriks attention berikut.
Gunakan bahasa Indonesia, ringkas, dan kaitkan dengan Query, Key, Value.

Token: {tokens}
Attention weights: {attention_summary}
"""

print(call_gemini_text(llm_prompt))